# Payment Transaction Request Test
This notebook sends a STATUS CHECK request to the ICICI UAT API.

In [ ]:
import requests
import json

In [ ]:
# Define the API endpoint for ICICI STATUS CHECK
API_URL = "https://pgpayuat.icicibank.com/tsp/pg/api/command"

# Define the request payload
payload = {
    "merchantId": "100000000007164",
    "aggregatorID" :"A100000000007164", 
    "merchantTxnNo": "G43UGDD9B5EKIIFZT328",
    "originalTxnNo": "7575obsaa7575",
    "transactionType": "STATUS",
    "secureHash":"18da3c1e40e150e6d4c703ee18c2c72750bd128132015ea6f68fbda17bf8d20f"
}

# Define headers
headers = {
    "Content-Type": "application/json"
}

print("Payload being sent:")
print(json.dumps(payload, indent=2))

Payload being sent:
{
  "merchantId": "100000000007164",
  "aggregatorID": "A100000000007164",
  "merchantTxnNo": "7575obsaa7575",
  "originalTxnNo": "7575obsaa7575",
  "transactionType": "STATUS",
  "secureHash": "18da3c1e40e150e6d4c703ee18c2c72750bd128132015ea6f68fbda17bf8d20f"
}


In [11]:
# Send the POST request
try:
    response = requests.post(API_URL, json=payload, headers=headers)

    print(f"Status Code: {response.status_code}")
    print(f"\nResponse:")
    print(json.dumps(response.json(), indent=2))

except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Status Code: 200

Response:
{
  "secureHash": "ee918e5f2e413b25e1763157620c6c1858ba048ea28acf61bb76f07554fa3988",
  "merchantId": "100000000007164",
  "respDescription": "Invalid request: Secure hash does not match",
  "merchantTxnNo": "7575obsaa7575",
  "responseCode": "P1006"
}


In [5]:
from pathlib import Path
import os
import hmac
import hashlib
import json
import requests

def load_env_file(env_path):
    env_path = Path(env_path)
    if not env_path.exists():
        raise FileNotFoundError(f"{env_path} not found")

    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)

load_env_file("backend/.env")

ICICI_HMAC_SECRET = os.getenv("ICICI_HMAC_SECRET")
if not ICICI_HMAC_SECRET:
    raise ValueError("ICICI_HMAC_SECRET is not set in backend/.env")

API_URL = "https://pgpayuat.icicibank.com/tsp/pg/api/command"

payload_without_hash = {
    "merchantId": "100000000007164",
    "aggregatorID": "A100000000007164",
    "merchantTxnNo": "G43UGDD9B5EKIIFZT328",
    "originalTxnNo": "G43UGDD9B5EKIIFZT328",
    "transactionType": "STATUS",
}

concatenated_values = "".join(
    str(payload_without_hash[key] or "")
    for key in sorted(payload_without_hash)
)

secure_hash = hmac.new(
    ICICI_HMAC_SECRET.encode("utf-8"),
    concatenated_values.encode("utf-8"),
    hashlib.sha256,
).hexdigest()

payload = {
    **payload_without_hash,
    "secureHash": secure_hash,
}

print("Concatenated values used for hash:")
print(concatenated_values)
print("\nComputed secureHash:")
print(secure_hash)
print("\nPayload being sent:")
print(json.dumps(payload, indent=2))

Concatenated values used for hash:
A100000000007164100000000007164G43UGDD9B5EKIIFZT328G43UGDD9B5EKIIFZT328STATUS

Computed secureHash:
87462a7eec6af53f2811a75533741668a6bd3d358d89afa0fd3e79d087279a7a

Payload being sent:
{
  "merchantId": "100000000007164",
  "aggregatorID": "A100000000007164",
  "merchantTxnNo": "G43UGDD9B5EKIIFZT328",
  "originalTxnNo": "G43UGDD9B5EKIIFZT328",
  "transactionType": "STATUS",
  "secureHash": "87462a7eec6af53f2811a75533741668a6bd3d358d89afa0fd3e79d087279a7a"
}


In [6]:
headers = {
    "Content-Type": "application/json"
}

# Send the POST request
try:
    response = requests.post(API_URL, json=payload, headers=headers)

    print(f"Status Code: {response.status_code}")
    print(f"\nResponse:")
    print(json.dumps(response.json(), indent=2))

except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Status Code: 200

Response:
{
  "txnRespDescription": "Transaction successful",
  "secureHash": "5d1fb25554e8f5f68df90498207392fafffdd4877b07d062b413e6e7f9f5cfaf",
  "amount": "85.00",
  "txnAuthID": "91090374870",
  "txnResponseCode": "0000",
  "customerEmailID": "es22btech11001@iith.ac.in",
  "paymentMode": "Card",
  "respDescription": "Request processed successfully",
  "aggregatorID": "A100000000007164",
  "TransmissionDateTime": "20260402161021",
  "oth_charge": "2.34",
  "paymentInstId": "XXXX XXXX XXXX 0035",
  "responseCode": "000",
  "transactionType": "SALE",
  "acqName": "PayPhi",
  "cardNetwork": "VISA",
  "txnStatus": "SUC",
  "paymentSubInstType": "CC",
  "merchantId": "100000000007164",
  "merchantTxnNo": "G43UGDD9B5EKIIFZT328",
  "paymentDateTime": "20260402161750",
  "txnID": "7700226711118"
}
